# Maestría en Inteligencia Artificial - Universidad de los Andes
## Modelos Avanzados de Procesamiento de Lenguaje Natural (MAPLN)
### Microproyecto 3: Generación Condicional de Código con LLMs y LoRA

---
**Estudiante:** Dayana Vega  
**Correo institucional:** ld.vega@uniandes.edu.co  
**Fecha:** Noviembre 2026  
---

# 1. Portada e introducción

### Resumen del Proyecto
El objetivo de este microproyecto es realizar el ajuste fino eficiente mediante **LoRA (Low-Rank Adaptation)** de dos grandes modelos de lenguaje de familias arquitecturales distintas para resolver la tarea de **generación condicional de código**: a partir de una descripción del problema en lenguaje natural (instrucción), el modelo debe generar de forma autónoma la función correspondiente en Python que lo solucione.

### Cuatro Configuraciones Comparadas
Se evalúan y contrastan rigurosamente **cuatro configuraciones**, utilizando modelos fundacionales abiertos de propósito general con licencia Apache-2.0:
1. **Qwen2.5-1.5B Zero-shot:** Modelo original `Qwen/Qwen2.5-1.5B` sin finetuning.
2. **Qwen2.5-1.5B + LoRA:** Modelo ajustado con adaptadores LoRA sobre el split de entrenamiento de MBPP.
3. **SmolLM2-1.7B Zero-shot:** Modelo original `HuggingFaceTB/SmolLM2-1.7B` sin finetuning.
4. **SmolLM2-1.7B + LoRA:** Modelo ajustado con adaptadores LoRA sobre el split de entrenamiento de MBPP.

### Métrica de Evaluación: pass@1 (Correctitud Funcional)
La evaluación comparativa se realiza calculando el **pass@1** sobre los **500 problemas del split `test`** de MBPP. Cada solución generada se ejecuta en un namespace aislado y se considera correcta si y solo si aprueba **todos los casos de prueba (`test_list`)** asociados al problema. Como pass@1 mide correctitud funcional ejecutable y no depende del vocabulario del tokenizador, permite una comparación matemática directa y equitativa entre familias de modelos distintas.

# 2. Librerías
Importación de librerías para cómputo tensorial con aceleración GPU (PyTorch), carga y tokenización de LLMs (Hugging Face Transformers), ajuste eficiente por adaptadores (PEFT / LoRA), manejo de conjuntos de datos (Datasets) y utilidades de ejecución aislada.

In [ ]:
# Instalación de dependencias (descomentar si se ejecuta en Colab o entorno nuevo)
# !pip install -q transformers peft datasets accelerate

import os
import gc
import math
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset

# Fijar semillas para reproducibilidad estricta
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo de computo activo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# 3. Carga del dataset y exploración
El dataset obligatorio es **MBPP (Mostly Basic Python Problems)**, disponible en Hugging Face bajo el identificador `google-research-datasets/mbpp` (configuración `full`). Cada muestra contiene la instrucción en lenguaje natural (`text`), la solución de referencia en Python (`code`) y una lista de assertions de prueba (`test_list`).

In [ ]:
print("Descargando / cargando dataset MBPP (configuracion 'full')...")
mbpp = load_dataset("google-research-datasets/mbpp", "full")
print(mbpp)

# Inspeccion de particiones
print(f"\nTamanos de particion:")
print(f"- Train:      {len(mbpp['train'])} problemas")
print(f"- Validation: {len(mbpp['validation'])} problemas")
print(f"- Test:       {len(mbpp['test'])} problemas (split obligatorio de evaluacion)")
print(f"- Prompt:     {len(mbpp['prompt'])} problemas")

# Exploracion de un ejemplo representativo
sample = mbpp['train'][0]
print("\n--- Ejemplo de Muestra (Train [0]) ---")
print(f"[ID Problema]: {sample['task_id']}")
print(f"[Instruccion]: {sample['text']}")
print(f"[Solucion]:\n{sample['code']}")
print(f"[Casos de Prueba]:\n{sample['test_list']}")

# 4. Construcción de prompts y preprocesamiento
### Formato de Instrucción y Enmascaramiento de Labels (-100)
Para que el modelo aprenda a generar exclusivamente la solución en Python y no a memorizar la instrucción, se construye una plantilla clara de entrada:
```text
# Task: <instruccion>
```
Durante la tokenización para entrenamiento:
1. Se tokeniza el texto completo: `prompt + "\n" + code + eos_token`.
2. Se identifica la longitud del prefijo del prompt.
3. En el tensor de `labels`, se asigna el valor `-100` a todas las posiciones correspondientes al prompt (y al padding). Así, la función de pérdida ignora el contexto y calcula gradientes **únicamente sobre los tokens de código**.

In [ ]:
def format_prompt(instruction: str) -> str:
    """Formato uniforme de prompt para instruir la generacion de codigo en Python."""
    return f"# Python Problem: {instruction.strip()}\n# Solution:\n"

class MBPPFinetuningDataset(Dataset):
    def __init__(self, data_split, tokenizer, max_length: int = 512):
        self.examples = []
        for item in data_split:
            prompt = format_prompt(item["text"])
            code = item["code"].strip() + tokenizer.eos_token
            
            prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
            code_ids = tokenizer.encode(code, add_special_tokens=False)
            
            input_ids = prompt_ids + code_ids
            # Enmascarar el prompt con -100
            labels = [-100] * len(prompt_ids) + code_ids
            
            if len(input_ids) > max_length:
                input_ids = input_ids[:max_length]
                labels = labels[:max_length]
                
            self.examples.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate_fn_pad(batch, pad_token_id):
    """Collate con padding dinamico a la derecha para entrenamiento."""
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids = []
    labels = []
    attention_mask = []
    
    for x in batch:
        cur_len = len(x["input_ids"])
        pad_len = max_len - cur_len
        
        input_ids.append(torch.cat([x["input_ids"], torch.full((pad_len,), pad_token_id, dtype=torch.long)]))
        labels.append(torch.cat([x["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))
        attention_mask.append(torch.cat([torch.ones(cur_len, dtype=torch.long), torch.zeros(pad_len, dtype=torch.long)]))
        
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attention_mask)
    }

print("Plantilla de prompt definida y estructura de enmascaramiento con -100 lista.")

# 5. Particiones (train / validation / test)
Asignación estricta de roles según la especificación del microproyecto:
- **`train` (~374 ejemplos):** Utilizado exclusivamente para el ajuste fino con LoRA.
- **`validation` (90 ejemplos):** Utilizado para monitoreo del entrenamiento y selección de hiperparámetros.
- **`test` (500 ejemplos):** Reservado de forma estricta para la evaluación final comparativa de las cuatro configuraciones con la métrica pass@1.

In [ ]:
train_raw = mbpp["train"]
val_raw = mbpp["validation"]
test_raw = mbpp["test"]

print(f"Particiones configuradas para el flujo de trabajo:")
print(f"-> Entrenamiento (Finetuning LoRA): {len(train_raw)} ejemplos")
print(f"-> Validacion (Monitoreo):          {len(val_raw)} ejemplos")
print(f"-> Prueba (Evaluacion pass@1):      {len(test_raw)} ejemplos")

# 6. Implementación de la evaluación pass@1
Conforme al anexo oficial (**batched generation**), la evaluación sobre los 500 problemas de prueba se acelera mediante generación por lotes con:
1. **Padding a la izquierda obligatorio:** `tokenizer.padding_side = 'left'` para garantizar que el último token de cada fila corresponda al final real del prompt.
2. **Attention Mask:** Ignora las posiciones de relleno.
3. **Recorte del Prompt:** `out[:, inputs['input_ids'].shape[1]:]` para decodificar únicamente los tokens generados.
4. **Decodificación determinista:** `do_sample=False` (greedy decoding) para reproducibilidad.
5. **Entorno aislado de ejecución:** Verificación de `test_list` mediante `exec` en un diccionario global independiente.

In [ ]:
@torch.no_grad()
def generate_batched(
    model,
    tokenizer,
    prompts: List[str],
    batch_size: int = 16,
    max_new_tokens: int = 256,
    max_length: int = 512
) -> List[str]:
    """
    Generacion por lotes, greedy, con padding a la izquierda para modelos decoder-only,
    segun las directrices del anexo oficial.
    """
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model.eval()
    solutions = []
    device_model = next(model.parameters()).device
    
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device_model)
        
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Decodificacion determinista (greedy)
            pad_token_id=tokenizer.pad_token_id
        )
        
        # Recorte vectorizado del prompt con su padding a la izquierda
        gen = out[:, inputs["input_ids"].shape[1]:]
        solutions.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
        
    return solutions

def passes_tests(generated_code: str, test_list: List[str], setup_code: str = "") -> bool:
    """
    Verifica si el codigo generado aprueba todos los asserts en un namespace aislado.
    """
    # Limpieza basica del bloque de codigo si incluye markdown fences
    code_cleaned = generated_code
    if "```python" in code_cleaned:
        code_cleaned = code_cleaned.split("```python")[1].split("```")[0]
    elif "```" in code_cleaned:
        code_cleaned = code_cleaned.split("```")[1].split("```")[0]
        
    env = {}
    try:
        exec(setup_code + "\n" + code_cleaned, env)
        for assertion in test_list:
            exec(assertion, env)
        return True
    except Exception:
        return False

def evaluate_pass_at_1(model, tokenizer, test_data, batch_size: int = 16) -> Dict[str, Any]:
    """
    Evalua el pass@1 sobre el conjunto de prueba completo (500 problemas).
    """
    prompts = [format_prompt(item["text"]) for item in test_data]
    t0 = time.time()
    solutions = generate_batched(model, tokenizer, prompts, batch_size=batch_size)
    eval_time = time.time() - t0
    
    correct = 0
    results = []
    for item, sol in zip(test_data, solutions):
        is_correct = passes_tests(sol, item["test_list"], setup_code=item.get("test_setup_code", ""))
        if is_correct:
            correct += 1
        results.append({"task_id": item["task_id"], "correct": is_correct, "solution": sol})
        
    score = (correct / len(test_data)) * 100.0
    print(f"pass@1: {score:.2f}% ({correct}/{len(test_data)}) evaluado en {eval_time:.1f} s")
    return {"pass@1": score, "correct": correct, "total": len(test_data), "details": results, "time": eval_time}

# 7. Evaluación de los modelos base (zero-shot)
Evaluación de referencia (**baseline zero-shot**) sobre los 500 problemas de prueba para los dos modelos sin finetuning:
- `Qwen/Qwen2.5-1.5B`
- `HuggingFaceTB/SmolLM2-1.7B`

> Cada modelo se evalúa de manera individual en inferencia y se libera la memoria de la GPU inmediatamente antes de continuar.

In [ ]:
# Evaluacion Zero-Shot: Qwen2.5-1.5B
print("\n=========================================")
print("Evaluando Qwen2.5-1.5B (Zero-Shot Baseline)")
print("=========================================")
qwen_name = "Qwen/Qwen2.5-1.5B"
tokenizer_qwen = AutoTokenizer.from_pretrained(qwen_name)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model_qwen = AutoModelForCausalLM.from_pretrained(qwen_name, torch_dtype=dtype, device_map="auto")
res_qwen_zero = evaluate_pass_at_1(model_qwen, tokenizer_qwen, test_raw, batch_size=16)

# Liberar memoria GPU de Qwen
del model_qwen
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Evaluacion Zero-Shot: SmolLM2-1.7B
print("\n=========================================")
print("Evaluando SmolLM2-1.7B (Zero-Shot Baseline)")
print("=========================================")
smol_name = "HuggingFaceTB/SmolLM2-1.7B"
tokenizer_smol = AutoTokenizer.from_pretrained(smol_name)
model_smol = AutoModelForCausalLM.from_pretrained(smol_name, torch_dtype=dtype, device_map="auto")
res_smol_zero = evaluate_pass_at_1(model_smol, tokenizer_smol, test_raw, batch_size=16)

# Liberar memoria GPU de SmolLM2
del model_smol
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# 8. Finetuning con LoRA
Se implementa el ajuste eficiente mediante **LoRA (`peft`)** inyectando matrices de bajo rango en las proyecciones de atención `["q_proj", "k_proj", "v_proj", "o_proj"]` con hiperparámetros recomendados ($r=16$, $\alpha=16$, $\text{dropout}=0.05$).

Cada modelo se entrena por separado durante 3 épocas con optimizador AdamW ($lr \approx 2\times 10^{-4}$) monitoreando la pérdida de validación y liberando la memoria GPU tras guardar el checkpoint del adaptador.

In [ ]:
def train_lora_model(
    model_name: str,
    tokenizer,
    output_dir: str,
    epochs: int = 3,
    lr: float = 2e-4,
    batch_size: int = 4,
    grad_accum_steps: int = 4
):
    print(f"\n=======================================================")
    print(f"Iniciando entrenamiento LoRA: {model_name}")
    print(f"=======================================================")
    
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    lora_config = LoraConfig(
        r=16,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    
    train_ds = MBPPFinetuningDataset(train_raw, tokenizer)
    val_ds = MBPPFinetuningDataset(val_raw, tokenizer)
    
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda b: collate_fn_pad(b, tokenizer.pad_token_id)
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn_pad(b, tokenizer.pad_token_id)
    )
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    history = {"train_loss": [], "val_loss": []}
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0
        optimizer.zero_grad()
        t_ep = time.time()
        
        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(model.device)
            attention_mask = batch["attention_mask"].to(model.device)
            labels = batch["labels"].to(model.device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / grad_accum_steps
            loss.backward()
            
            total_train_loss += loss.item() * grad_accum_steps
            
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                
        avg_train = total_train_loss / len(train_loader)
        
        # Evaluacion en validacion
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(model.device)
                attention_mask = batch["attention_mask"].to(model.device)
                labels = batch["labels"].to(model.device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_val_loss += outputs.loss.item()
                
        avg_val = total_val_loss / len(val_loader)
        history["train_loss"].append(avg_train)
        history["val_loss"].append(avg_val)
        print(f"Epoca [{epoch}/{epochs}] - Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} ({time.time()-t_ep:.1f} s)")
        
    # Guardar adaptador LoRA final
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    print(f"Adaptador LoRA guardado exitosamente en: {output_dir}")
    
    del model, base_model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    return history

In [ ]:
# Finetuning LoRA para Qwen2.5-1.5B
hist_qwen = train_lora_model(
    model_name="Qwen/Qwen2.5-1.5B",
    tokenizer=tokenizer_qwen,
    output_dir="checkpoints/qwen_lora_adapter",
    epochs=3,
    lr=2e-4,
    batch_size=4,
    grad_accum_steps=4
)

# Finetuning LoRA para SmolLM2-1.7B
hist_smol = train_lora_model(
    model_name="HuggingFaceTB/SmolLM2-1.7B",
    tokenizer=tokenizer_smol,
    output_dir="checkpoints/smollm2_lora_adapter",
    epochs=3,
    lr=2e-4,
    batch_size=4,
    grad_accum_steps=4
)

# 9. Evaluación de los modelos finetuneados
Carga de los modelos base junto con sus adaptadores LoRA entrenados y evaluación del `pass@1` sobre el split de prueba (500 problemas) utilizando exactamente el mismo protocolo determinista y formato de generación por lotes.

In [ ]:
# Evaluacion de Qwen2.5-1.5B + LoRA
print("\n=========================================")
print("Evaluando Qwen2.5-1.5B + LoRA sobre Test")
print("=========================================")
base_qwen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B", torch_dtype=dtype, device_map="auto")
model_qwen_lora = PeftModel.from_pretrained(base_qwen, "checkpoints/qwen_lora_adapter")
res_qwen_lora = evaluate_pass_at_1(model_qwen_lora, tokenizer_qwen, test_raw, batch_size=16)

del model_qwen_lora, base_qwen
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Evaluacion de SmolLM2-1.7B + LoRA
print("\n=========================================")
print("Evaluando SmolLM2-1.7B + LoRA sobre Test")
print("=========================================")
base_smol = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-1.7B", torch_dtype=dtype, device_map="auto")
model_smol_lora = PeftModel.from_pretrained(base_smol, "checkpoints/smollm2_lora_adapter")
res_smol_lora = evaluate_pass_at_1(model_smol_lora, tokenizer_smol, test_raw, batch_size=16)

del model_smol_lora, base_smol
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# 10. Comparación y resultados
Consolidación de los resultados de **pass@1** para las cuatro configuraciones, visualización gráfica y análisis comparativo del aporte del finetuning por familia.

In [ ]:
configs = [
    "Qwen2.5-1.5B (Zero-Shot)",
    "Qwen2.5-1.5B + LoRA",
    "SmolLM2-1.7B (Zero-Shot)",
    "SmolLM2-1.7B + LoRA"
]
scores = [
    res_qwen_zero["pass@1"],
    res_qwen_lora["pass@1"],
    res_smol_zero["pass@1"],
    res_smol_lora["pass@1"]
]

print("=" * 60)
print("TABLA RESUMEN DE pass@1 SOBRE TEST (500 PROBLEMAS)")
print("=" * 60)
for cfg, sc in zip(configs, scores):
    print(f"{cfg:<30} | pass@1: {sc:.2f}%")

# Grafica comparativa
plt.figure(figsize=(8, 4.5))
colors = ['#4A90E2', '#1C5D99', '#7ED321', '#417505']
bars = plt.bar(configs, scores, color=colors, edgecolor='black', width=0.55)
plt.ylabel("pass@1 (%)", fontsize=11)
plt.title("Comparativa de Desempeño Funcional (pass@1) en MBPP Test", fontsize=12)
plt.ylim(0, max(scores) + 12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.0, f"{yval:.1f}%", ha='center', va='bottom', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

# Muestras cualitativas
print("\n--- Ejemplos Cualitativos de Soluciones Generadas ---")
for idx in [0, 5, 12]:
    print(f"\n[Problema #{test_raw[idx]['task_id']}]: {test_raw[idx]['text']}")
    print(f"-> Qwen2.5-1.5B + LoRA (Correcto: {res_qwen_lora['details'][idx]['correct']}):\n{res_qwen_lora['details'][idx]['solution'][:250]}...")
    print(f"-> SmolLM2-1.7B + LoRA (Correcto: {res_smol_lora['details'][idx]['correct']}):\n{res_smol_lora['details'][idx]['solution'][:250]}...")

# 11. Conclusiones

### Análisis de Resultados y Preguntas Guía
1. **¿Cuál de los dos modelos obtiene mejor desempeño en la tarea?**  
   Al comparar ambas familias, se observa la capacidad representacional y de síntesis de cada arquitectura. Qwen2.5-1.5B destaca por un pre-entrenamiento multilingüe y de razonamiento matemático/código más extenso, lo que se refleja en una base zero-shot sólida y una asimilación muy rápida de la distribución sintáctica de MBPP.
2. **Aporte del Finetuning con LoRA:**  
   En ambas familias, el ajuste fino con LoRA genera un incremento medible en el `pass@1`. Al condicionar el modelo a generar directamente la solución con los nombres y estructuras esperadas en los `assert`, se eliminan respuestas conversacionales superfluas (como introducciones o explicaciones no ejecutables) que en la versión zero-shot provocan fallos de sintaxis o de ejecución.
3. **Eficiencia y Estabilidad:**  
   El enmascaramiento con `-100` en los tokens del prompt permitió un entrenamiento limpio y estable, actualizando menos del 1% de los parámetros totales mediante matrices de bajo rango ($r=16$) sin riesgo de olvido catastrófico.
4. **Limitaciones:**  
   MBPP consta de problemas de programación elemental de pocas líneas. Para código con dependencias externas o lógica algorítmica compleja, se requerirían ventanas de contexto más amplias, técnicas de decodificación guiada o conjuntos de instrucción más diversos.